# Groq API Authentication Smoke Test

**Purpose:**
This notebook verifies that the local Groq API key can authenticate and
complete one harmless API request.

**Scope:**
- No municipal zoning content is sent.
- No PDFs, source packets, manifests, or pilot outputs are read.
- No triage, extraction, scoring, or review workflow is run.
- This notebook is not part of the formal Stage 0–2 pilot sequence.


In [1]:
# --- Environment and safe key-presence check ---
# This cell never prints the key itself, a prefix/suffix of it, or any environment dump.
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / ".env").exists() and (REPO_ROOT.parent / ".env").exists():
    REPO_ROOT = REPO_ROOT.parent  # notebook may be launched from zoning_pdf_pilot/ itself

ENV_PATH = REPO_ROOT / ".env"
env_path_exists = ENV_PATH.exists()
if env_path_exists:
    load_dotenv(ENV_PATH)

groq_api_key = os.environ.get("GROQ_API_KEY")
key_present = bool(groq_api_key)
key_length = len(groq_api_key) if groq_api_key else 0

try:
    import importlib.metadata
    groq_version = importlib.metadata.version("groq")
except Exception:
    groq_version = "not_installed"

print(f".env path exists: {env_path_exists}")
print(f"GROQ_API_KEY present: {key_present}")
print(f"GROQ_API_KEY length: {key_length}")
print(f"Active Python executable: {sys.executable}")
print(f"Groq SDK version: {groq_version}")

if not key_present:
    print()
    print("GROQ_API_KEY is not present. Add it to the local .env file and rerun.")


.env path exists: True
GROQ_API_KEY present: True
GROQ_API_KEY length: 56
Active Python executable: /Users/sierramendoza/Documents/HAP-Repository/.venv/bin/python
Groq SDK version: 1.7.0


In [2]:
# --- One harmless Groq API request ---
# Model chosen from the SDK's own models.list() (not guessed): "allam-2-7b", a small,
# low-cost text model confirmed active on this account. No tools, no structured output,
# no system prompt, no zoning content. Exactly one request this run.

MODEL_NAME = "allam-2-7b"
HARMLESS_PROMPT = "Reply with exactly one word: OK"

print("=" * 72)
print("GROQ AUTH SMOKE TEST")
print("=" * 72)
print(f"Key present: {key_present}")
print(f"Key length: {key_length}")
print(f"Model requested: {MODEL_NAME}")

if not key_present:
    print("Request status: SKIPPED (no key present)")
else:
    from groq import Groq

    try:
        client = Groq(api_key=groq_api_key)
        completion = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[{"role": "user", "content": HARMLESS_PROMPT}],
        )
        bounded_response = (completion.choices[0].message.content or "")[:200]
        print("Request status: SUCCESS")
        print(f"Response: {bounded_response}")
    except Exception as exc:
        bounded_message = str(exc)[:500]
        print("Request status: FAILED")
        print(f"Exception class: {type(exc).__name__}")
        print(f"Message (truncated to 500 chars): {bounded_message}")

print("=" * 72)


GROQ AUTH SMOKE TEST
Key present: True
Key length: 56
Model requested: allam-2-7b


Request status: SUCCESS
Response: OK. 


---

A successful smoke test verifies local Groq authentication and basic API
connectivity only. It does not validate model quality, source-packet handling,
triage prompts, extraction logic, citation quality, or zoning conclusions.

No pilot packet should be sent to Groq until the Stage 3 triage design has
been reviewed and explicitly approved.
